# 2. 移動平均線

In [1]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

import plotly.graph_objs as go
import talib as ta
import datetime as dt
import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/.venv/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
sys.path.append("/workspace/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/workspace/data'))


In [3]:
name = "Nikkei 225"
code = "^N225"

# チャート表示開始日
display_start_date = dt.datetime(2023, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2024, 7, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### Nikkei 225 (^N225)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

df["ma5"] = ta.SMA(close, timeperiod=5)
df["ma25"] = ta.SMA(close, timeperiod=25)

# チャートに表示する期間
rdf = df[dt.datetime(2024, 1, 1):end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%Y-%m-%d')

# ハッシュ形式でレイアウトを指定 ----(1)
layout = {
    "height": 700,
    "width": 1000,
    "title": { "text": "日経平均株価", "x": 0.5 },
    "xaxis": { "rangeslider": { "visible": False }, "type": "category" },
    "yaxis1": { "domain": [.05, 1.0], "title": "株価(円)", "side": "left", "tickformat": "," },
    "yaxis2": { "domain": [.0, .05] },
}

# ローソク足や移動平均のデータをplotly.graph_objsライブラリのメソッドで用意 ----(2)
data = [
    go.Candlestick(yaxis="y1", x=rdf.index,
        open=rdf['Open'],
        high=rdf['High'],
        low=rdf['Low'],
        close=rdf['Close'],
        increasing_line_color='red',
        decreasing_line_color='gray',
        name="日経平均株価",
    ),
    go.Scatter(
        yaxis="y1",
        x=rdf.index,
        y=rdf['ma5'],
        name='MA5',
        line=dict(color='royalblue', width=1.2),
        
    ),
    go.Scatter(
        yaxis="y1",
        x=rdf.index,
        y=rdf['ma25'],
        name='MA25',
        line=dict(color='lightseagreen', width=1.5),
    )
]

# レイアウトの修正や表示 ----(3)
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示　スペース的に全部は表示できないので1/3にして表示
fig.update_layout({
    "xaxis": {
        "showgrid": False,
        "tickvals": rdf.index[::3],
        "ticktext": [" {}-{} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::3]],
    }
})

# グラフを表示
fig.show()

[*********************100%***********************]  1 of 1 completed


### 2.1.5 ゴールデンクロスとデッドクロス

In [4]:
name = "SoftBank Group"
code = "9984.T"

# チャート表示開始日
display_start_date = dt.datetime(2023, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2024, 7, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### SoftBank Group (9984.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

df["ma5"] = ta.SMA(close, timeperiod=5)
df["ma25"] = ta.SMA(close, timeperiod=25)

ma5, ma25 = df["ma5"], df["ma25"]
cross = ma5 > ma25

# crossを1日ずらしたシリーズを作成し、それぞれの発生日を一時的なシリーズに格納
cross_shift = cross.shift(1)
temp_gc = (cross != cross_shift) & (cross == True)
temp_dc = (cross != cross_shift) & (cross == False)

# 発生日に株価を格納してチャートを表示できるようにする ----(3)
# ゴールデンクロス発生日であればMA5の値、それ以外はNaN
gc = [m if g == True else np.nan for g, m in zip(temp_gc, ma5)]

# デッドクロス発生日であればMA5の値、それ以外はNaN
dc = [m if d == True else np.nan for d, m in zip(temp_dc, ma25)]

# データフレームにカラムとして保存
df["gc"] = gc
df["dc"] = dc

# チャートに表示する期間
rdf = df[dt.datetime(2024, 3, 1):end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%Y-%m-%d')

# ハッシュ形式でレイアウトを指定 ----(1)
layout = {
    "height": 700,
    "width": 1000,
    "title": { "text": "日経平均株価", "x": 0.5 },
    "xaxis": { "rangeslider": { "visible": False }, "type": "category" },
    "yaxis1": { "domain": [.05, 1.0], "title": "株価(円)", "side": "left", "tickformat": "," },
    "yaxis2": { "domain": [.0, .05] },
}

# ローソク足や移動平均のデータをplotly.graph_objsライブラリのメソッドで用意 ----(2)
data = [
    go.Candlestick(yaxis="y1", x=rdf.index,
        open=rdf['Open'],
        high=rdf['High'],
        low=rdf['Low'],
        close=rdf['Close'],
        increasing_line_color='red',
        decreasing_line_color='gray',
        name="日経平均株価",
    ),
    go.Scatter(
        yaxis="y1",
        x=rdf.index,
        y=rdf['ma5'],
        name='MA5',
        line=dict(color='royalblue', width=1.2),
        
    ),
    go.Scatter(
        yaxis="y1",
        x=rdf.index,
        y=rdf['ma25'],
        name='MA25',
        line=dict(color='lightseagreen', width=1.5),
    )
]

# レイアウトの修正や表示 ----(3)
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示　スペース的に全部は表示できないので1/3にして表示
fig.update_layout({
    "xaxis": {
        "showgrid": False,
        "tickvals": rdf.index[::3],
        "ticktext": [" {}-{} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::3]],
    }
})

# グラフを表示
fig.show()

[*********************100%***********************]  1 of 1 completed


### 2.1.6 移動平均とボリンジャーバンド

In [7]:
name = "SoftBank Group"
code = "9984.T"

# チャート表示開始日
display_start_date = dt.datetime(2023, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2024, 7, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### SoftBank Group (9984.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

df["ma5"] = ta.SMA(close, timeperiod=5)
df["ma25"] = ta.SMA(close, timeperiod=25)

ma5, ma25 = df["ma5"], df["ma25"]
cross = ma5 > ma25

# crossを1日ずらしたシリーズを作成し、それぞれの発生日を一時的なシリーズに格納
cross_shift = cross.shift(1)
temp_gc = (cross != cross_shift) & (cross == True)
temp_dc = (cross != cross_shift) & (cross == False)

# 発生日に株価を格納してチャートを表示できるようにする ----(3)
# ゴールデンクロス発生日であればMA5の値、それ以外はNaN
gc = [m if g == True else np.nan for g, m in zip(temp_gc, ma5)]

# デッドクロス発生日であればMA5の値、それ以外はNaN
dc = [m if d == True else np.nan for d, m in zip(temp_dc, ma25)]

# データフレームにカラムとして保存
df["gc"] = gc
df["dc"] = dc

# ボリンジャーバンドの算出
upper2, _, lower2 = ta.BBANDS(close, timeperiod=25, nbdevup=2, nbdevdn=2, matype=0)
df["upper2"], df["lower2"] = upper2, lower2

# チャートに表示する期間
rdf = df[dt.datetime(2024, 3, 1):end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%Y-%m-%d')

# ハッシュ形式でレイアウトを指定 ----(1)
layout = {
    "height": 700,
    "width": 1000,
    "title": { "text": "日経平均株価", "x": 0.5 },
    "xaxis": { "rangeslider": { "visible": False }, "type": "category" },
    "yaxis1": { "domain": [.05, 1.0], "title": "株価(円)", "side": "left", "tickformat": "," },
    "yaxis2": { "domain": [.0, .05] },
}

# データの設定
data = [
    go.Candlestick(yaxis="y1", x=rdf.index,
        open=rdf['Open'],
        high=rdf['High'],
        low=rdf['Low'],
        close=rdf['Close'],
        increasing_line_color='red',
        decreasing_line_color='gray',
        name=name,
    ),
    # 5日間移動平均線
    go.Scatter(yaxis="y1", x=rdf.index, y=rdf['ma5'],
                name='MA5', line={ "color": 'royalblue', "width": 1.2 },
        
    ),
    # 25日間移動平均線
    go.Scatter(yaxis="y1", x=rdf.index, y=rdf['ma25'],
                name='MA25', line={ "color": 'lightseagreen', "width": 1.5 },
    ),
    # ボリンジャーバンド
    go.Scatter(yaxis="y1", x=rdf.index, y=rdf['upper2'],
                name='', line={ "color": 'lavender', "width": 0 },
    ),
    go.Scatter(yaxis="y1", x=rdf.index, y=rdf['lower2'],
                name='BB', line={ "color": 'lavender', "width": 0 },
                fill='tonexty', fillcolor='rgba(170, 170, 170, 0.2)'
    )
]

# レイアウトの修正や表示 ----(3)
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示　スペース的に全部は表示できないので1/3にして表示
fig.update_layout({
    "xaxis": {
        "showgrid": False,
        "tickvals": rdf.index[::3],
        "ticktext": [" {}-{} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::3]],
    }
})

# グラフを表示
fig.show()

[*********************100%***********************]  1 of 1 completed


## 2.2 日足、週足、月足でに確認
### 2.2.1 週単位や月単位での集計

In [9]:
name = "Nikkei 225"
code = "^N225"

# チャート表示開始日
display_start_date = dt.datetime(2023, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2024, 7, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### Nikkei 225 (^N225)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)

# 日足データを週足データに変換
# resampleメソッドとaggregateメソッドを実行して、週足データフレームを作成
df_weekly = df.resample('W').aggregate({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

close = df_weekly['Close']

df_weekly

[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Date,,,,,
2022-03-13,25634.089844,25720.310547,24681.740234,25162.779297,496400000
2022-03-20,25338.640625,26862.429688,25219.130859,26827.429688,425900000
2022-03-27,27091.320312,28338.810547,27076.330078,28149.839844,325200000
2022-04-03,28084.080078,28281.449219,27399.480469,27665.980469,371800000
2022-04-10,27685.650391,27965.939453,26764.359375,26985.800781,332500000
...,...,...,...,...,...
2024-07-07,39839.820312,41100.128906,39457.621094,40912.371094,611700000
2024-07-14,40863.140625,42426.769531,40780.699219,41190.679688,619000000
2024-07-21,41366.789062,41520.070312,39824.578125,40063.789062,428100000


### 2.2.2 週足と月足の移動平均線の表示

In [10]:
# 移動平均線の算出
df_weekly["ma5"] = ta.SMA(close, timeperiod=5)
df_weekly["ma25"] = ta.SMA(close, timeperiod=25)

# ボリンジャーバンドの算出
upper2, _, lower2 = ta.BBANDS(close, timeperiod=25, nbdevup=2, nbdevdn=2, matype=0)
df_weekly["upper2"], df_weekly["lower2"] = upper2, lower2

# 2023.4.1から2024.7.31までのチャートを作成
df_weekly = df_weekly[dt.datetime(2023, 4, 1):dt.datetime(2024, 7, 31)]
df_weekly.index = pd.to_datetime(df_weekly.index).strftime('%Y-%m-%d')

# ハッシュ形式でレイアウトを指定 ----(1)
layout = {
    "height": 700,
    "width": 1000,
    "title": { "text": "日経平均株価", "x": 0.5 },
    "xaxis": { "rangeslider": { "visible": False }, "type": "category" },
    "yaxis1": { "domain": [.05, 1.0], "title": "株価(円)", "side": "left", "tickformat": "," },
    "yaxis2": { "domain": [.0, .05] },
}

# データの設定
data = [
    go.Candlestick(yaxis="y1", x=df_weekly.index,
        open=df_weekly['Open'],
        high=df_weekly['High'],
        low=df_weekly['Low'],
        close=df_weekly['Close'],
        increasing_line_color='red',
        decreasing_line_color='gray',
        name=name,
    ),
    # 5日間移動平均線
    go.Scatter(yaxis="y1", x=df_weekly.index, y=df_weekly['ma5'],
                name='MA5', line={ "color": 'royalblue', "width": 1.2 },
        
    ),
    # 25日間移動平均線
    go.Scatter(yaxis="y1", x=df_weekly.index, y=df_weekly['ma25'],
                name='MA25', line={ "color": 'lightseagreen', "width": 1.5 },
    ),
    # ボリンジャーバンド
    go.Scatter(yaxis="y1", x=df_weekly.index, y=df_weekly['upper2'],
                name='', line={ "color": 'lavender', "width": 0 },
    ),
    go.Scatter(yaxis="y1", x=df_weekly.index, y=df_weekly['lower2'],
                name='BB', line={ "color": 'lavender', "width": 0 },
                fill='tonexty', fillcolor='rgba(170, 170, 170, 0.2)'
    )
]

# レイアウトの修正や表示 ----(3)
fig = go.Figure(data=data, layout=go.Layout(layout))

# 日付表示　スペース的に全部は表示できないので1/3にして表示
fig.update_layout({
    "xaxis": {
        "showgrid": False,
        "tickvals": df_weekly.index[::3],
        "ticktext": [" {}-{} ".format(x.split("-")[0], x.split("-")[1]) for x in df_weekly.index[::3]],
    }
})

# グラフを表示
fig.show()


### 2.2.3 経済イベントとの関連の確認

In [15]:
name = "Nikkei 225"
code = "^N225"

# チャート表示開始日
display_start_date = dt.datetime(2024, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2024, 12, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### Nikkei 225 (^N225)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

# インデクスを文字列型に
df.index = pd.to_datetime(df.index).strftime('%Y-%m-%d')

# 日銀政策決定会合の日付
boj_meetings = [
    { 'date': '2024-01-23', 'event': '金融政策決定会合' },
    { 'date': '2024-03-19', 'event': '金融政策決定会合' },
    { 'date': '2024-04-26', 'event': '金融政策決定会合' },
    { 'date': '2024-06-14', 'event': '金融政策決定会合' },
    { 'date': '2024-07-31', 'event': '金融政策決定会合' },
    { 'date': '2024-09-20', 'event': '金融政策決定会合' },
    { 'date': '2024-10-31', 'event': '金融政策決定会合' },
    { 'date': '2024-12-19', 'event': '金融政策決定会合' },
]

# チャートに表示する期間
rdf = df[display_start_date.strftime('%Y-%m-%d'):end_date.strftime('%Y-%m-%d')]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%Y-%m-%d')


# レイアウト設定
layout = {
          "title"  : { "text": "日経平均株価", "x":0.5 },
          "xaxis" : { "rangeslider": { "visible": True }, "type": "category" },
          "template": "plotly_white",
        }

# データの設定
data =  [
      go.Candlestick(yaxis="y1", x=rdf.index,
                        open=rdf['Open'],
                        high=rdf['High'],
                        low=rdf['Low'],
                        close=rdf['Close'],
                        increasing_line_color="red", 
                        decreasing_line_color="gray",
                        name='日経平均株価',
                        hovertext=rdf.apply(
                            lambda row: f"日付: {row.name}<br>"
                                        f"Open: {int(row['Open'])}<br>"
                                        f"High: {int(row['High'])}<br>"
                                        f"Low: {int(row['Low'])}<br>"
                                        f"Close: {int(row['Close'])}", axis=1
                        ),
                        hoverinfo="text"  # ホバーテキストのみを表示
        ),

]

# Figure作成
fig = go.Figure(data=data, layout=go.Layout(layout))

# 金融政策決定会合日の注釈と縦線を追加
for meeting in boj_meetings:
    # 点線を追加
    fig.add_shape(
        type="line",
        x0=meeting['date'],
        y0=rdf['Low'].min(),
        x1=meeting['date'],
        y1=rdf['High'].max() * 1.05,
        line=dict(color="blue", dash="dash")
    )
    # 注釈を追加
    fig.add_annotation(
        x=meeting['date'],
        y=rdf['High'].max() * 1.05,
        text=meeting['event'],
        showarrow=False,
        yanchor="bottom",
        font=dict(size=10, color="blue")
    )


# 日付表示
fig.update_layout({
"xaxis":{
    "showgrid": False,
    "tickvals": rdf.index[::5],
    "ticktext": ["{}-{}".format(x.split("-")[1], x.split("-")[2]) for x in rdf.index[::5]]
    }
})

# グラフを表示
fig.show()

[*********************100%***********************]  1 of 1 completed